# Bayesian M2M Gaussian experiment

This notebook is only the experiment driver and plotting layer.

The computational stages are separate scripts:

1. Generate and freeze \(D=\{(X^i,Y^i)\}_{i=1}^n\) and a separate unseen \((X^0,Y^0)\).
2. Call `collect_pca_trajectory_fixed.py` to pretrain and construct the PCA subspace from this fixed \(D\).
3. Call `run_ess.py` to sample the posterior PCA coordinates \(\phi\).
4. Call `posterior_predictive.py` to evaluate the posterior predictive at \(X^0\).

The notebook produces:
- a pair plot of the \(K\) posterior coordinates \(\phi\);
- the posterior predictive distribution of the target mean \(m_\phi(X^0)\).


In [ ]:
!git clone https://github.com/djskog/simple_gaussian_example.git/

In [ ]:
!git -C /content/simple_gaussian_example pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 335 bytes | 335.00 KiB/s, done.
From https://github.com/djskog/simple_gaussian_example
   1d1b62c..52ec80b  main       -> origin/main
Updating 1d1b62c..52ec80b
Fast-forward
 plot_predictions.py | 4 +++-
 1 file changed, 3 insertions(+), 1 deletion(-)


In [ ]:
from pathlib import Path
import sys
import subprocess
import random

import numpy as np
import torch
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/simple_gaussian_example"
)

# ------------------------------------------------------------
# Project setup
# ------------------------------------------------------------

# In Colab, make sure the notebook is running from the repository root.
%cd /content/simple_gaussian_example
PROJECT_ROOT = Path.cwd()

print("Working directory:", PROJECT_ROOT)

# Check that the project files are actually here.
required_files = [
    "datasets.py",
    "utils.py",
    "collect_pca_trajectory.py",
    "run_ess.py",
    "posterior_predictive.py",
]

missing = [f for f in required_files if not (PROJECT_ROOT / f).exists()]

if missing:
    raise FileNotFoundError(
        "Missing project files:\n" + "\n".join(f"  - {f}" for f in missing)
    )

# Make local project modules importable.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from datasets import generate_location_data
from utils import bayes_optimal_target_mean, get_df_label

# Output directories
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
FIGURE_DIR = DRIVE_ROOT / "figures"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Drive root:", DRIVE_ROOT)
print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Experiment configuration

SEED = 7

N_TRAIN_MEASURES = 500
N_MEASURES = 10
N_POINTS = 128

beta = torch.zeros(2)
Sigma_Z = torch.eye(2)
Sigma_X = 0.25 * torch.eye(2)
Sigma_Y = 0.10 * torch.eye(2)
B0 = torch.eye(2)

T_DFs = [np.inf, 3.0, 4.0, 5.0, 8.0, 12.0, 20.0, 50.0]  

# PCA trajectory
PRETRAIN_EPOCHS = 50
PRETRAIN_LR = 1e-3
TRAJECTORY_EPOCHS = 50
TRAJECTORY_LR = 1e-2
MAX_SNAPSHOTS = 20
PCA_RANK = 5
BATCH_SIZE = 64

# ESS
PRIOR_STD = 1
ESS_BURN_IN = 100
ESS_NUM_SAMPLES = 1000
ESS_THIN = 1
ESS_TEMPERATURE = 1e3
ESS_SEED = 1234

## 1. Generate and freeze \(D\) and an unseen test pair

We materialize the observations once. The later PCA and ESS scripts use these exact tensors, so the likelihood is deterministic conditional on the fixed dataset.


In [ ]:
#generate and save D
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

for df in T_DFs:
    data = generate_location_data(
        num_measures=N_TRAIN_MEASURES,
        num_points=N_POINTS,
        df=df,
        beta=beta,
        Sigma_Z=Sigma_Z,
        Sigma_X=Sigma_X,
        Sigma_Y=Sigma_Y,
        B0=B0,
        seed=SEED,
    )
    
    path = (CHECKPOINT_DIR / f"{get_df_label(df)}_fixed_D.pt")
    torch.save(data, path)
    print("Saved:", path)




In [ ]:
#generate and save X0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

test_data = generate_location_data(
        num_measures=N_MEASURES,
        num_points=N_POINTS,
        df=np.inf,  # Gaussian case
        beta=beta,
        Sigma_Z=Sigma_Z,
        Sigma_X=Sigma_X,
        Sigma_Y=Sigma_Y,
        B0=B0,
        seed=SEED,
    )

path = (CHECKPOINT_DIR / f"{get_df_label(np.inf)}_fixed_data.pt")
torch.save(test_data, path)
print("Saved:", path)

## 2. Construct the PCA subspace

The separate collector starts from a fresh transformer, pretrains on the fixed \(D\), continues with constant-step-size SGD, forms the SWA shift \(\hat w\), and computes the scaled PCA basis \(P\), giving

\[
w(\phi)=\hat w+P\phi.
\]


In [ ]:
pca_script = PROJECT_ROOT / "collect_pca_trajectory.py"

if not pca_script.exists():
    raise FileNotFoundError(f"Missing {pca_script}")

for df in T_DFs:
    train_data_path = CHECKPOINT_DIR / f"{get_df_label(df)}_fixed_D.pt"
    pca_path = CHECKPOINT_DIR / f"{get_df_label(df)}_pca_subspace.pt"

    cmd = [
        sys.executable,
        "-u",
        str(pca_script),
        "--fixed-data", str(train_data_path),
        "--output", str(pca_path),
        "--pretrain-epochs", str(PRETRAIN_EPOCHS),
        "--pretrain-lr", str(PRETRAIN_LR),
        "--trajectory-epochs", str(TRAJECTORY_EPOCHS),
        "--trajectory-lr", str(TRAJECTORY_LR),
        "--max-snapshots", str(MAX_SNAPSHOTS),
        "--pca-rank", str(PCA_RANK),
        "--batch-size", str(BATCH_SIZE),
        "--seed", str(SEED),
    ]

    print("Running PCA collector...", flush=True)

    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(
            return_code,
            cmd,
        )

    print(f"{get_df_label(df)} PCA collector finished.", flush=True)


In [ ]:
##load PCA checkpoint and print info

pca_checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=False,
)

print("PCA basis shape:", tuple(pca_checkpoint["pca_basis"].shape))
print("Explained variance ratio:")
print(pca_checkpoint["explained_variance_ratio"])


## 3. Sample \(\phi\mid D\) with elliptical slice sampling

The ESS script targets

\[
p_T(\phi\mid D)
\propto
p(D\mid\phi)^{1/T}p(\phi),
\qquad
\phi\sim N(0,I).
\]

All likelihood evaluations use the frozen dataset \(D\).


In [ ]:
ess_script = PROJECT_ROOT / "run_ess.py"
if not ess_script.exists():
    raise FileNotFoundError(f"Missing {ess_script}")

for df in T_DFs:
    train_data_path = CHECKPOINT_DIR / f"{get_df_label(df)}_fixed_D.pt"
    pca_path = CHECKPOINT_DIR / f"{get_df_label(df)}_pca_subspace.pt"
    posterior_path = CHECKPOINT_DIR / f"{get_df_label(df)}_posterior_samples.pt"

    cmd = [
        sys.executable,
        "-u",
        str(ess_script),
        "--fixed-data", str(train_data_path),
        "--pca", str(pca_path),
        "--output", str(posterior_path),
        "--prior-std", str(PRIOR_STD),
        "--burn-in", str(ESS_BURN_IN),
        "--num-samples", str(ESS_NUM_SAMPLES),
        "--thin", str(ESS_THIN),
        "--temperature", str(ESS_TEMPERATURE),
        "--batch-size", str(BATCH_SIZE),
        "--seed", str(ESS_SEED),
    ]

    print("Running ESS...", flush=True)

    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(
            return_code,
            cmd,
        )


## 4. Pair plot of the \(K\) posterior PCA coordinates

Diagonal panels show marginal histograms. Off-diagonal panels show posterior scatter plots.


In [ ]:
for df in T_DFs:

    # -------------------------------------------------------------
    # Load posterior samples
    # -------------------------------------------------------------

    posterior_path = (
        CHECKPOINT_DIR
        / f"{get_df_label(df)}_posterior_samples.pt"
    )

    if not posterior_path.exists():
        raise FileNotFoundError(
            f"Missing posterior file: {posterior_path}"
        )

    posterior = torch.load(
        posterior_path,
        map_location="cpu",
        weights_only=False,
    )

    phi_samples = posterior[
        "phi_samples"
    ].numpy()

    K = phi_samples.shape[1]

    # -------------------------------------------------------------
    # Pair plot
    # -------------------------------------------------------------

    fig, axes = plt.subplots(
        K,
        K,
        figsize=(2.1 * K, 2.1 * K),
        squeeze=False,
    )

    for i in range(K):
        for j in range(K):

            ax = axes[i, j]

            if i == j:

                ax.hist(
                    phi_samples[:, i],
                    bins=35,
                    alpha=0.75,
                )

            else:

                ax.scatter(
                    phi_samples[:, j],
                    phi_samples[:, i],
                    s=5,
                    alpha=0.20,
                )

            if i == K - 1:
                ax.set_xlabel(
                    rf"$\phi_{j+1}$"
                )
            else:
                ax.set_xticklabels([])

            if j == 0:
                ax.set_ylabel(
                    rf"$\phi_{i+1}$"
                )
            else:
                ax.set_yticklabels([])

            ax.grid(
                alpha=0.15
            )

    fig.suptitle(
        rf"Posterior samples in PCA coordinates ($K={K}$)"
        f"\n{get_df_label(df)}",
        y=0.995,
    )

    plt.tight_layout()

    # -------------------------------------------------------------
    # Save
    # -------------------------------------------------------------

    figure_path = (
        FIGURE_DIR
        / f"{get_df_label(df)}_phi_pairplot.png"
    )

    fig.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight",
    )

    print(
        f"Saved: {figure_path}"
    )

    plt.show()
    plt.close(fig)


## 5. Posterior predictive for the unseen source \(X^0\)

For every posterior sample,

\[
\phi^{(s)}\sim p(\phi\mid D),
\qquad
m_s=m_{\phi^{(s)}}(X^0).
\]

These \(m_s\) are samples from the posterior distribution of the target measure's mean.


In [ ]:
predictive_script = PROJECT_ROOT / "posterior_predictive.py"

if not predictive_script.exists():
    raise FileNotFoundError(
        f"Missing {predictive_script}"
    )

for df in T_DFs:
    pca_path = CHECKPOINT_DIR / f"{get_df_label(df)}_pca_subspace.pt"
    posterior_path = CHECKPOINT_DIR / f"{get_df_label(df)}_posterior_samples.pt"
    test_path = CHECKPOINT_DIR / f"gaussian_fixed_test_data.pt"
    predictive_path = CHECKPOINT_DIR / f"{get_df_label(df)}_posterior_predictive.pt"

    cmd = [
        sys.executable,
        "-u",
        str(predictive_script),
        "--pca", str(pca_path),
        "--posterior", str(posterior_path),
        "--test-data", str(test_path),
        "--output", str(predictive_path),
        "--num-predictive-models",
        str(min(500, ESS_NUM_SAMPLES)),
        "--seed",
        str(ESS_SEED + 1),
    ]

    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(
            return_code,
            cmd,
        )

In [ ]:
n_plots = 10

plot_script = PROJECT_ROOT / "plot_predictions.py"

if not plot_script.exists():
    raise FileNotFoundError(
        f"Missing {plot_script}"
    )

cmd = [
    sys.executable,
    "-u",
    str(plot_script),
    "--checkpoint-dir",
    str(CHECKPOINT_DIR),
    "--figure-dir",
    str(FIGURE_DIR),
    "--n-plots",
    str(n_plots),
    "--bins",
    "40",
    "--dfs",
    *[str(df) for df in T_DFs],
]

print(
    "Running predictive df-sweep plotting...",
    flush=True,
)

process = subprocess.Popen(
    cmd,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None

for line in process.stdout:
    print(
        line,
        end="",
        flush=True,
    )

return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(
        return_code,
        cmd,
    )

print(
    "Predictive plotting finished.",
    flush=True,
)

Running predictive df-sweep plotting...


CalledProcessError: Command '['/usr/bin/python3', '-u', '/content/simple_gaussian_example/plot_predictions.py', '--checkpoint-dir', '/content/drive/MyDrive/simple_gaussian_example/checkpoints', '--figure-dir', '/content/drive/MyDrive/simple_gaussian_example/figures', '--n-plots', '10', '--bins', '40', '--dfs', 'inf', '3.0', '4.0', '5.0', '8.0', '12.0', '20.0', '50.0']' returned non-zero exit status 1.

## Output files

The experiment leaves these reusable files in `checkpoints/`:

- `fixed_D_and_X0.pt` — frozen training dataset and unseen test pair;
- `pca_subspace_fixed_D.pt` — SWA point and PCA subspace;
- `posterior_phi.pt` — ESS samples of \(\phi\);
- `posterior_predictive_X0.pt` — posterior samples of the target mean and one predictive point-cloud realization.
